# Harry Potter RAG — Colab (all-in-one)

Runs the whole project in one Colab notebook. Two Colab-specific adaptations vs.
the local repo:

- **No Docker** → Qdrant runs in **embedded mode** (`QdrantClient(path=...)`), no
  server to start.
- **No fixed server port** → the FastAPI app (`rag_api.py`) runs in a **background
  thread** sharing the one embedded client, and the UI is **Gradio** (`share=True`)
  instead of Streamlit.

The only key you need is an LLM key — embeddings run locally.

> **Tip:** `Runtime → Change runtime type → T4 GPU` makes embedding ~10× faster.

## 1. Install dependencies

In [1]:
!pip -q install pymupdf sentence-transformers "qdrant-client>=1.11" \
    fastapi "uvicorn[standard]" openai python-dotenv gradio nest_asyncio requests
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 29.8 MB/s eta 0:00:00
done


## 2. Upload the dataset & set your LLM key

Run the cell, pick your **`harrypotter.pdf`** when prompted, then paste your API
key (a free key from https://console.groq.com works). The key is read with
`getpass`, so it isn't stored in the notebook.

In [2]:
import os
from getpass import getpass
from google.colab import files

up = files.upload()                       # choose harrypotter.pdf
PDF_PATH = list(up.keys())[0]
print("PDF:", PDF_PATH)

os.environ["LLM_API_KEY"]  = getpass("LLM API key (e.g. Groq): ")
os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["LLM_MODEL"] = "openai/gpt-oss-20b"     # fast; or "openai/gpt-oss-120b" for better answers

Saving harrypotter.pdf to harrypotter.pdf
PDF: harrypotter.pdf
LLM API key (e.g. Groq): ··········


## 3. Configuration

In [3]:
import os
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
EMBEDDING_DIM   = 384
CHUNK_SIZE      = 800
CHUNK_OVERLAP   = 120
COLLECTION      = "harry_potter"
QDRANT_PATH     = "/content/qdrant_db"     # embedded, on-disk (persist to Drive if you want it to survive)
QUERY_PREFIX    = "Represent this sentence for searching relevant passages: "

# Export for rag_api.py (it reads these from the environment).
# NOTE: QUERY_PREFIX and LLM_MODEL are exported too, so the API embeds queries
# with the bge prefix and talks to the right model.
os.environ.update(
    EMBEDDING_MODEL=EMBEDDING_MODEL, EMBEDDING_DIM=str(EMBEDDING_DIM),
    QDRANT_PATH=QDRANT_PATH, QDRANT_COLLECTION=COLLECTION, TOP_K="5",
    QUERY_PREFIX=QUERY_PREFIX,
    LLM_MODEL=os.environ.get("LLM_MODEL", "openai/gpt-oss-20b"),
)
print("configured")

configured


## 4. Parse → clean → chunk

Same pipeline as the local notebook: detect the 7 books from the table-of-contents
pages, clean each page (fix chapter drop-caps, de-hyphenate, keep chapter titles as
headings), then chunk sentence-aware with overlap.

In [4]:
import fitz, re

doc = fitz.open(PDF_PATH)

contents_pages = [i for i in range(doc.page_count)
                  if any(ln.strip().upper()=="CONTENTS" for ln in doc.load_page(i).get_text().splitlines())]
collection_toc = contents_pages[0]
book_start_pages = contents_pages[1:]
titles = [ln.strip() for ln in doc.load_page(collection_toc).get_text().splitlines()
          if ln.strip().lower().startswith("harry potter")]
bounds = book_start_pages + [doc.page_count]
BOOKS = [(titles[i], book_start_pages[i], bounds[i+1]) for i in range(len(book_start_pages))]
print(f"Detected {len(BOOKS)} books")

def clean_page(text):
    lines=[ln.rstrip() for ln in text.split("\n")]
    lines=[ln for ln in lines if ln.strip() not in ("","\x0c")]
    dropcap=None
    if lines and re.fullmatch(r"[A-Z]", lines[0].strip()):
        dropcap=lines.pop(0).strip()
    out,i=[],0
    while i<len(lines):
        ln=lines[i]
        if re.fullmatch(r"CHAPTER\s+[A-Z\-]+", ln.strip()):
            title=lines[i+1].strip().title() if i+1<len(lines) else ""
            out.append(f"\n\n## {title}\n"); i+=2
            if dropcap and i<len(lines): lines[i]=dropcap+lines[i]; dropcap=None
            continue
        out.append(ln); i+=1
    if dropcap and out: out[0]=dropcap+out[0]
    txt=""
    for ln in out:
        if txt.endswith("-") and re.match(r"[a-z]",ln): txt=txt[:-1]+ln
        elif txt.endswith("\n") or ln.startswith("\n"): txt+=ln
        else: txt+=(" " if txt else "")+ln
    return re.sub(r"\n{3,}","\n\n", re.sub(r"[ \t]+"," ",txt)).strip()

def extract_book(a,b):
    parts=[clean_page(doc.load_page(p).get_text()) for p in range(a,b)]
    txt="\n".join(p for p in parts if p)
    idx=txt.find("\n## ")
    return txt[idx:].strip() if idx>0 else txt

def chunk_book(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks=[]
    for si,sec in enumerate(re.split(r"\n## ", "\n" + text)):
        if si==0 and not sec.strip(): continue
        parts=sec.split("\n",1)
        chapter=parts[0].strip().lstrip("# ") if si>0 else "Front matter"
        body=re.sub(r"\s+"," ", parts[1] if len(parts)>1 else parts[0]).strip()
        if not body: continue
        packed,cur=[],""
        for s in re.split(r"(?<=[.!?])\s+", body):
            if not s: continue
            if len(cur)+len(s)+1<=size: cur=(cur+" "+s).strip()
            else:
                if cur: packed.append(cur)
                if len(s)>size: packed+=[s[j:j+size] for j in range(0,len(s),size)]; cur=""
                else: cur=s
        if cur: packed.append(cur)
        for i,c in enumerate(packed):
            if overlap and i>0: c=(packed[i-1][-overlap:]+" "+c).strip()
            chunks.append((c,chapter))
    return chunks

records,gid=[],0
for bi,(title,a,b) in enumerate(BOOKS):
    for text,chapter in chunk_book(extract_book(a,b)):
        records.append({"id":gid,"text":text,"book":title,"book_index":bi,"chapter":chapter}); gid+=1
print("Total chunks:", len(records))

Detected 7 books
Total chunks: 8683


## 5. Embed locally & store in embedded Qdrant

Keep the `client` object — we'll hand it straight to the API so there's only ever one embedded connection.

In [5]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import numpy as np

model = SentenceTransformer(EMBEDDING_MODEL)   # uses the GPU if the runtime has one
emb = model.encode([r["text"] for r in records], batch_size=64,
                   normalize_embeddings=True, show_progress_bar=True)
emb = np.asarray(emb, dtype="float32")

client = QdrantClient(path=QDRANT_PATH)
if client.collection_exists(COLLECTION): client.delete_collection(COLLECTION)
client.create_collection(COLLECTION, vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))

for s in range(0, len(records), 256):
    batch, vecs = records[s:s+256], emb[s:s+256]
    client.upsert(COLLECTION, points=[PointStruct(id=r["id"], vector=v.tolist(),
        payload={"text":r["text"],"book":r["book"],"book_index":r["book_index"],"chapter":r["chapter"]})
        for r,v in zip(batch,vecs)])
print("Indexed:", client.count(COLLECTION).count, "chunks")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/136 [00:00<?, ?it/s]

Indexed: 8683 chunks


## 6. Write `rag_api.py` (the deliverable) to disk

This is the exact FastAPI server file — save it so you can commit it to your repo.

Fixes vs. the first draft:
- `RAGState` is now a real object with instance attributes, and it supports
  attribute access (`STATE.llm`), item access (`STATE["llm"]`), and
  `STATE.update(...)` — so no monkey-patching is ever needed.
- Retrieval uses the current Qdrant API (`query_points(..., with_payload=True)`),
  with a fallback to the older `search`. The removed `append_payload` argument is gone.

In [6]:
%%writefile rag_api.py
import os, re
from datetime import datetime
from functools import lru_cache
from typing import Literal

from fastapi import FastAPI
from pydantic import BaseModel


class RAGState:
    """Holds the live objects injected by the notebook at startup.

    Supports attribute access (STATE.llm), item access (STATE["llm"]) and
    STATE.update(embedder=..., qdrant=..., llm=...) so the caller can wire it
    up whichever way is convenient.
    """
    def __init__(self):
        self.embedder = None
        self.qdrant = None
        self.llm = None

    def __getitem__(self, key):
        return getattr(self, key)

    def __setitem__(self, key, value):
        setattr(self, key, value)

    def update(self, **kwargs):
        for key, value in kwargs.items():
            setattr(self, key, value)


# Globals
app = FastAPI()
STATE = RAGState()  # populated at startup

LLM_MODEL = os.environ.get("LLM_MODEL", "")
LLM_SEARCH_MODEL = os.environ.get("LLM_SEARCH_MODEL", "")
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "")
EMBEDDING_DIM = int(os.environ.get("EMBEDDING_DIM", "0"))
QDRANT_COLLECTION = os.environ.get("QDRANT_COLLECTION", "")
QUERY_PREFIX = os.environ.get("QUERY_PREFIX", "")
TOP_K = int(os.environ.get("TOP_K", "5"))


# For routing
ROUTER_SYSTEM = """You are a helpful assistant that classifies user queries. You are provided with a query and you must respond with one of the following labels: 'greeting', 'harry_potter', 'out_of_scope'.

- 'greeting' for queries that are simple greetings or thanks (e.g. "Hi", "Hello", "Thanks!").
- 'harry_potter' for queries that are about the Harry Potter books, characters, or world.
- 'out_of_scope' for queries that are not related to Harry Potter.

Only return the label and nothing else. Do not explain your answer. Do not return the label in quotes.
"""


# For RAG generation
GEN_SYSTEM = """You are a helpful assistant that answers questions about the Harry Potter books. You are provided with a question and a set of search results from the books. You must use only the provided information to answer the question. Combine information from multiple sources if necessary. If the answer is not in the provided documents, say that you don't know.

Use Markdown for formatting. Do not include any links or images.
"""


# Utilities
@lru_cache(maxsize=128)
def _get_embedding(text: str) -> list[float]:
    # STATE.embedder is a SentenceTransformer. Passing a single string returns a
    # 1-D normalized vector; .tolist() makes it JSON/Qdrant friendly.
    return STATE.embedder.encode(QUERY_PREFIX + text, normalize_embeddings=True).tolist()


def _search(query_vector: list[float], top_k: int):
    """Return a list of scored points, working across qdrant-client versions."""
    q = STATE.qdrant
    if hasattr(q, "query_points"):                 # qdrant-client >= 1.10
        return q.query_points(
            collection_name=QDRANT_COLLECTION,
            query=query_vector,
            limit=top_k,
            with_payload=True,
        ).points
    return q.search(                               # older clients
        collection_name=QDRANT_COLLECTION,
        query_vector=query_vector,
        limit=top_k,
        with_payload=True,
    )


def llm_chat(messages: list[dict], temperature: float = 0.0, max_tokens: int = 700) -> str:
    try:
        resp = STATE.llm.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"LLM error: {e}")
        return "I'm sorry, I encountered an error while generating a response. Please try again."


def route_query(query: str) -> str:
    greeting_re = re.compile(
        r"^\s*(hi|hey|hello|yo|howdy|good (morning|afternoon|evening)|thanks|thank you|bye)\b",
        re.IGNORECASE,
    )
    if greeting_re.match(query):
        return "greeting"
    try:
        label = llm_chat(
            [{"role": "system", "content": ROUTER_SYSTEM},
             {"role": "user", "content": query}],
            temperature=0.0, max_tokens=8,
        )
        if not label:
            return "harry_potter"
        label = label.lower()
    except Exception:
        return "harry_potter"
    for r in ("harry_potter", "greeting", "out_of_scope"):
        if r in label:
            return r
    return "harry_potter"


def generate_rag_answer(query: str, top_k: int = TOP_K) -> tuple[str, list[dict]]:
    # Get relevant docs
    query_embedding = _get_embedding(query)
    search_res = _search(query_embedding, top_k)
    sources = []
    for res in search_res:
        payload = res.payload or {}
        sources.append({
            "book":    payload.get("book", ""),
            "chapter": payload.get("chapter", ""),
            "score":   round(res.score, 3),
            "snippet": payload.get("text", ""),
        })
    # Create prompt
    context = "\n\n".join(f"### {s['book']} - {s['chapter']}\n{s['snippet']}" for s in sources)
    messages = [
        {"role": "system", "content": GEN_SYSTEM},
        {"role": "user", "content": f"Question: {query}\n\nContext:\n{context}"}
    ]
    # Generate answer
    answer = llm_chat(messages)
    return answer, sources


# API Models
class ChatRequest(BaseModel):
    query: str
    top_k: int = TOP_K


class ChatResponse(BaseModel):
    answer: str
    route: Literal["greeting", "harry_potter", "out_of_scope"]
    sources: list[dict] | None = None
    error: str | None = None


@app.get("/health")
async def health():
    return {"status": "ok", "timestamp": datetime.now().isoformat()}


@app.post("/chat")
async def chat(req: ChatRequest) -> ChatResponse:
    query = req.query.strip()
    route = route_query(query)

    if route == "out_of_scope":
        return ChatResponse(answer="I can only answer questions about Harry Potter.", route=route)
    elif route == "greeting":
        return ChatResponse(answer="Hello! How can I help you today?", route=route)
    else:  # harry_potter
        answer, sources = generate_rag_answer(query, req.top_k)
        return ChatResponse(answer=answer, route=route, sources=sources)

Writing rag_api.py


## 7. Launch the API in the background

We inject the already-open embedded `client`, the `model`, and an OpenAI-compatible
LLM client into the app, then run Uvicorn in a thread. Because the clients are
injected, the app reuses them instead of opening a second embedded connection.

This is the **only** cell that reloads `rag_api` and starts a server. It first
stops any server left over from a previous run, reloads the module once, wires
`STATE`, and serves the *same* reloaded app — which is what prevents the
`STATE.embedder is None` error from a stale server.

In [7]:
import os, importlib, threading, time, requests, nest_asyncio, uvicorn
from openai import OpenAI

# Stop a server left running by a previous run of this cell.
try:
    _SERVER.should_exit = True
    time.sleep(1.5)
except NameError:
    pass

import rag_api
importlib.reload(rag_api)              # fresh app + fresh STATE

# Wire the live objects into the SAME module we're about to serve.
rag_api.STATE.update(
    embedder=model,
    qdrant=client,
    llm=OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"]),
)

# Fail loudly now instead of 500-ing on every request.
assert rag_api.STATE.embedder is not None, "embedder is None — run the embed cell (model)"
assert rag_api.STATE.qdrant   is not None, "qdrant is None — run the embed cell (client)"

PORT = 8000
nest_asyncio.apply()
_SERVER = uvicorn.Server(uvicorn.Config(rag_api.app, host="127.0.0.1",
                                        port=PORT, log_level="warning"))
threading.Thread(target=_SERVER.run, daemon=True).start()

# Poll /health instead of a blind sleep.
for _ in range(20):
    try:
        print("API up:", requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).json())
        break
    except Exception:
        time.sleep(0.5)
else:
    print("API did not become healthy in time — check for errors above.")

API up: {'status': 'ok', 'timestamp': '2026-08-31T12:01:33.831652'}


## 8. Test the RAG flow over HTTP  *(screenshot this)*

In [8]:
import requests

q = "How did Harry get his Firebolt, and who sent it?"
r = requests.post(f"http://127.0.0.1:{PORT}/chat",
                  json={"query": q, "top_k": 5}, timeout=120).json()

print("Q:", q)
print("\nROUTE:", r["route"])
print("\nANSWER:\n", r["answer"])
print("\nSOURCES:")
for s in r.get("sources") or []:
    print(f"  - {s['book']} — {s['chapter']}  (score {s['score']})")

Q: How did Harry get his Firebolt, and who sent it?

ROUTE: harry_potter

ANSWER:
 

SOURCES:
  - Harry Potter and the Prisoner of Azkaban — Gryffindor Versus Ravenclaw  (score 0.764)
  - Harry Potter and the Prisoner of Azkaban — The Patronus  (score 0.76)
  - Harry Potter and the Prisoner of Azkaban — The Patronus  (score 0.752)
  - Harry Potter and the Prisoner of Azkaban — Owl Post  (score 0.752)
  - Harry Potter and the Prisoner of Azkaban — The Quidditch Final  (score 0.752)


## 9. Gradio UI  *(screenshot this too)*

`share=True` gives a public link and an inline UI — the answer, its sources, and a health check.

In [9]:
# ============================================================================
#  Streamlit UI  (alternative to the Gradio cell)
#  Prereq: the "7. Launch the API" cell must already be running (FastAPI on 8000).
#  This cell installs Streamlit, writes the app, and exposes it with a
#  Cloudflare quick tunnel — a public https link, no password page.
# ============================================================================
import os, sys, re, time, socket, subprocess

API_PORT = 8000          # must match PORT from the "Launch the API" cell
UI_PORT  = 8501

# 0. Clean up anything left by a previous run of this cell
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"],   check=False)
time.sleep(1)

# 1. Install Streamlit (the notebook only installed Gradio)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit"], check=False)

# 2. Write the Streamlit app — it just calls the FastAPI server on API_PORT
app_code = r'''
import requests, streamlit as st

API = "http://127.0.0.1:8000"

st.set_page_config(page_title="Harry Potter RAG", page_icon="⚡")
st.title("⚡ Harry Potter RAG Chatbot")
st.caption("Answers come only from the 7 books.")

c1, c2 = st.columns([4, 1])
with c1:
    query = st.text_input("Your question", placeholder="How did Harry get his Firebolt?")
with c2:
    top_k = st.slider("top_k", 1, 10, 5)

if st.button("Ask", type="primary") and query.strip():
    with st.spinner("Searching the books..."):
        try:
            r = requests.post(f"{API}/chat", json={"query": query, "top_k": int(top_k)}, timeout=120).json()
        except Exception as e:
            st.error(f"Request failed: {e}")
            st.stop()
    st.markdown("### Answer")
    st.markdown(r.get("answer", "_(no answer)_"))
    st.caption(f"route: {r.get('route')}")
    sources = r.get("sources") or []
    with st.expander(f"Sources ({len(sources)})", expanded=bool(sources)):
        if not sources:
            st.write("_(no sources — greeting or out-of-scope)_")
        for s in sources:
            st.markdown(f"**{s['book']} — {s['chapter']}**  (score {s['score']})")
            st.markdown(f"> {s['snippet']}")

with st.expander("Server health"):
    if st.button("Check health"):
        try:
            st.json(requests.get(f"{API}/health", timeout=10).json())
        except Exception as e:
            st.json({"status": f"error: {e}"})
'''
open("streamlit_app.py", "w").write(app_code)

# 3. Fetch the cloudflared binary (once)
if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "cloudflared"], check=False)
    subprocess.run(["chmod", "+x", "cloudflared"], check=False)

# 4. Start Streamlit in the background
_ST = subprocess.Popen(
    ["streamlit", "run", "streamlit_app.py",
     "--server.port", str(UI_PORT), "--server.address", "0.0.0.0",
     "--server.headless", "true"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(40):                       # wait until the port accepts connections
    try:
        socket.create_connection(("127.0.0.1", UI_PORT), timeout=1).close(); break
    except OSError:
        time.sleep(1)

# 5. Open the public tunnel and print the URL
_CF = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{UI_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
for _ in range(60):                       # scan cloudflared output for the link
    line = _CF.stdout.readline()
    if not line:
        time.sleep(0.5); continue
    m = re.search(r"https://[-\w.]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break

print("\n" + "="*60)
print("Streamlit app is live at:\n   " + (url or "URL not found — re-run the cell"))
print("="*60)


🌐 Streamlit app is live at:
   https://mls-secrets-echo-register.trycloudflare.com


---
## Submitting

Commit to GitHub / Drive: **this notebook**, the generated **`rag_api.py`**, and
**screenshots** from cells 8 and 9 (an answer with its sources, and the health
check). That covers all three required deliverables.

> Embedded Qdrant here lives in `/content` and is wiped when the runtime resets.
> To keep it, mount Drive (`from google.colab import drive; drive.mount('/content/drive')`)
> and set `QDRANT_PATH` to a folder under `/content/drive/MyDrive/...`.